# 01 - Read ClinVar `variant_summary`

Notebook nay doc file ClinVar `variant_summary.txt.gz` bang pandas va kiem tra cac cot chinh can cho cac buoc phan tich tiep theo.

Neu trong project chua co file `.gz`, notebook se tu fallback sang `Data/variant_summary.txt`.

In [ ]:
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

PROJECT_DIR = Path("/mnt/MyData/Bioinformatics/Project")
DATA_DIR = PROJECT_DIR / "Data"

candidate_paths = [
    DATA_DIR / "variant_summary.txt.gz",
    DATA_DIR / "variant_summary.txt",
]

variant_path = next((path for path in candidate_paths if path.exists()), None)
if variant_path is None:
    raise FileNotFoundError(
        "Khong tim thay /mnt/MyData/Bioinformatics/Project/Data/variant_summary.txt.gz "
        "hoac /mnt/MyData/Bioinformatics/Project/Data/variant_summary.txt"
    )

print(f"Project dir: {PROJECT_DIR}")
print(f"ClinVar file: {variant_path}")

variant_path

## Doc header bang pandas

File ClinVar variant summary rat lon, nen buoc dau chi doc header (`nrows=0`) de lay danh sach cot.

In [ ]:
header_df = pd.read_csv(
    variant_path,
    sep="\t",
    dtype=str,
    nrows=0,
    compression="infer",
)

columns = header_df.columns.tolist()
columns

## Kiem tra cac cot chinh

In [ ]:
required_columns = [
    "ClinicalSignificance",
    "Assembly",
    "Chromosome",
    "PositionVCF",
    "ReferenceAlleleVCF",
    "AlternateAlleleVCF",
    "Type",
    "ReviewStatus",
    "GeneSymbol",
]

column_check = pd.DataFrame(
    {
        "column": required_columns,
        "exists": [column in columns for column in required_columns],
    }
)

missing_columns = column_check.loc[~column_check["exists"], "column"].tolist()
display(column_check)

if missing_columns:
    raise ValueError(f"Thieu cot bat buoc: {missing_columns}")

print("Tat ca cot chinh deu ton tai.")

## Doc du lieu ClinVar bang pandas

De giam RAM, cell nay chi doc cac cot chinh. Neu can doc them cot, them ten cot vao `use_columns`.

In [ ]:
use_columns = required_columns

clinvar_df = pd.read_csv(
    variant_path,
    sep="\t",
    dtype=str,
    usecols=use_columns,
    compression="infer",
)

clinvar_df.shape

In [ ]:
clinvar_df.head()

## Kiem tra nhanh gia tri trong cac cot chinh

In [ ]:
summary = pd.DataFrame(
    {
        "non_null": clinvar_df.notna().sum(),
        "null": clinvar_df.isna().sum(),
        "unique": clinvar_df.nunique(dropna=True),
    }
).loc[required_columns]

summary

In [ ]:
for column in ["ClinicalSignificance", "Assembly", "Type", "ReviewStatus"]:
    print(f"\n{column}")
    display(clinvar_df[column].value_counts(dropna=False).head(20))

## 2. Loc nhan du doan

Chi giu cac bien the co `ClinicalSignificance` ro rang:

- `Pathogenic`
- `Likely pathogenic`
- `Benign`
- `Likely benign`

Loai bo cac nhan mo ho nhu `Uncertain significance`, `Conflicting interpretations`, `not provided`, va cac nhan khac khong nam trong bon nhom tren.

Sau do gop nhan:

- `Pathogenic` / `Likely pathogenic` -> `Y = 1`
- `Benign` / `Likely benign` -> `Y = 0`

Muc tieu la tao bien `Y` sach cho supervised learning.

In [ ]:
positive_labels = {"Pathogenic", "Likely pathogenic"}
negative_labels = {"Benign", "Likely benign"}


def split_clinical_significance(value):
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split("/") if part.strip()]


def map_clean_label(value):
    labels = split_clinical_significance(value)
    label_set = set(labels)

    if not labels:
        return pd.NA
    if label_set <= positive_labels:
        return 1
    if label_set <= negative_labels:
        return 0
    return pd.NA


clinvar_labeled_df = clinvar_df.copy()
clinvar_labeled_df["Y"] = clinvar_labeled_df["ClinicalSignificance"].map(map_clean_label)

clinvar_clean_y_df = clinvar_labeled_df.dropna(subset=["Y"]).copy()
clinvar_clean_y_df["Y"] = clinvar_clean_y_df["Y"].astype("int8")

print(f"Tong so dong ban dau: {len(clinvar_df):,}")
print(f"So dong giu lai co nhan Y sach: {len(clinvar_clean_y_df):,}")
print(f"So dong bi loai bo do nhan mo ho: {len(clinvar_df) - len(clinvar_clean_y_df):,}")

display(
    clinvar_clean_y_df["Y"]
    .value_counts()
    .rename(index={1: "Pathogenic/Likely pathogenic", 0: "Benign/Likely benign"})
)

In [ ]:
excluded_label_counts = (
    clinvar_labeled_df.loc[clinvar_labeled_df["Y"].isna(), "ClinicalSignificance"]
    .value_counts(dropna=False)
    .head(20)
)

excluded_label_counts

## 3. Loc theo he gen tham chieu

Chi giu cac dong co:

- `Assembly = GRCh38`

Dataframe sau buoc nay la `clinvar_clean_df`.

In [ ]:
clinvar_clean_df = clinvar_clean_y_df.loc[
    clinvar_clean_y_df["Assembly"].eq("GRCh38")
].copy()

print(f"So dong co nhan Y sach truoc khi loc Assembly: {len(clinvar_clean_y_df):,}")
print(f"So dong sau khi giu Assembly = GRCh38: {len(clinvar_clean_df):,}")
print(f"So dong bi loai do khac GRCh38: {len(clinvar_clean_y_df) - len(clinvar_clean_df):,}")

display(clinvar_clean_df["Assembly"].value_counts(dropna=False))
display(
    clinvar_clean_df["Y"]
    .value_counts()
    .rename(index={1: "Pathogenic/Likely pathogenic", 0: "Benign/Likely benign"})
)

## 4. Loc loai dot bien

O giai doan dau, chi giu:

- `Type = single nucleotide variant`

Tuc la chi lay SNP / dot bien diem.

Tam thoi loai bo insertion, deletion, duplication, indel dai, copy number variation, structural variant.

Ly do: CNN can input co chieu dai co dinh; neu dua ca mat doan/them doan dai thi sequence window se kho chuan hoa.

In [ ]:
clinvar_snv_df = clinvar_clean_df.loc[
    clinvar_clean_df["Type"].eq("single nucleotide variant")
].copy()

print(f"So dong truoc khi loc Type: {len(clinvar_clean_df):,}")
print(f"So dong sau khi giu Type = single nucleotide variant: {len(clinvar_snv_df):,}")
print(f"So dong bi loai do khac SNV: {len(clinvar_clean_df) - len(clinvar_snv_df):,}")

display(clinvar_snv_df["Type"].value_counts(dropna=False))
display(
    clinvar_clean_df.loc[~clinvar_clean_df["Type"].eq("single nucleotide variant"), "Type"]
    .value_counts(dropna=False)
    .head(20)
)

## 5. Loc chat luong nhan bang ReviewStatus

Loai bo cac dong co do tin cay thap, vi du:

- `no assertion criteria provided`

Uu tien giu cac dong co `ReviewStatus` chua mot trong cac cum:

- `criteria provided`
- `reviewed by expert panel`
- `practice guideline`

Muc tieu la giam label noise, vi mo hinh hoc tu nhan sai thi ket qua cung sai.

In [ ]:
trusted_review_patterns = [
    "criteria provided",
    "reviewed by expert panel",
    "practice guideline",
]

review_status = clinvar_snv_df["ReviewStatus"].fillna("")
trusted_review_mask = review_status.str.contains(
    "|".join(trusted_review_patterns),
    case=False,
    regex=True,
)

# Loai rieng nhom rat yeu, du phong neu sau nay pattern bi thay doi.
low_confidence_review_mask = review_status.str.fullmatch(
    "no assertion criteria provided",
    case=False,
)

clinvar_model_df = clinvar_snv_df.loc[
    trusted_review_mask & ~low_confidence_review_mask
].copy()

print(f"So dong truoc khi loc ReviewStatus: {len(clinvar_snv_df):,}")
print(f"So dong sau khi giu ReviewStatus dang tin cay: {len(clinvar_model_df):,}")
print(f"So dong bi loai do ReviewStatus yeu/mo ho: {len(clinvar_snv_df) - len(clinvar_model_df):,}")

display(clinvar_model_df["ReviewStatus"].value_counts(dropna=False).head(20))
display(
    clinvar_snv_df.loc[~(trusted_review_mask & ~low_confidence_review_mask), "ReviewStatus"]
    .value_counts(dropna=False)
    .head(20)
)

## 6. Loai dong thieu du lieu quan trong

Bo cac dong bi thieu o cac cot can dung cho nhan, toa do, allele, gene va cac bo loc chinh.

In [ ]:
required_cols = [
    "ClinicalSignificance",
    "Assembly",
    "Chromosome",
    "PositionVCF",
    "ReferenceAlleleVCF",
    "AlternateAlleleVCF",
    "Type",
    "ReviewStatus",
    "GeneSymbol",
]

clinvar_complete_df = clinvar_model_df.dropna(subset=required_cols).copy()

# ClinVar hay dung "-" de bieu dien gia tri khong co/khong ap dung.
missing_token_mask = clinvar_complete_df[required_cols].isin(["", "-", "na", "NA"]).any(axis=1)
clinvar_complete_df = clinvar_complete_df.loc[~missing_token_mask].copy()

print(f"So dong truoc khi loai missing: {len(clinvar_model_df):,}")
print(f"So dong sau khi loai missing: {len(clinvar_complete_df):,}")
print(f"So dong bi loai do thieu du lieu quan trong: {len(clinvar_model_df) - len(clinvar_complete_df):,}")

clinvar_complete_df[required_cols].isna().sum()

## 7. Chuan hoa chromosome

Chuan hoa ten chromosome de khop voi FASTA.

- Neu FASTA dung `chr1`, `chr2`, `chrX` thi doi `1 -> chr1`, `2 -> chr2`, `X -> chrX`.
- Neu FASTA dung `1`, `2`, `X` thi giu nguyen.

Cot moi `ChromosomeFASTA` se duoc dung khi truy xuat bang `pyfaidx`.

In [30]:
FASTA_PATH = PROJECT_DIR / "Data" / "ncbi_dataset" / "ncbi_dataset" / "data" / "GCF_000001405.26" / "GCF_000001405.26_GRCh38_genomic.fna"

if not FASTA_PATH.exists():
    raise FileNotFoundError(f"Khong tim thay FASTA GRCh38: {FASTA_PATH}")


def read_fasta_sequence_names(fasta_path, max_records=500):
    names = []
    with open(fasta_path, "rt") as handle:
        for line in handle:
            if line.startswith(">"):
                names.append(line[1:].split()[0])
                if len(names) >= max_records:
                    break
    return names


fasta_sequence_names = read_fasta_sequence_names(FASTA_PATH)
fasta_sequence_name_set = set(fasta_sequence_names)

uses_chr_prefix = any(name.startswith("chr") for name in fasta_sequence_names)

# NCBI RefSeq FASTA dung accession nhu NC_000001.11. Map chromosome ClinVar sang accession GRCh38.
chromosome_to_refseq = {str(i): f"NC_{i:06d}.11" for i in range(1, 23)}
chromosome_to_refseq.update({"X": "NC_000023.11", "Y": "NC_000024.10", "MT": "NC_012920.1", "M": "NC_012920.1"})


def normalize_chromosome_for_fasta(chromosome):
    chrom = str(chromosome).strip()
    if chrom.lower().startswith("chr"):
        chrom = chrom[3:]
    chrom = "MT" if chrom in {"M", "MT", "m", "mt"} else chrom.upper()

    direct_candidates = [str(chromosome).strip(), chrom, f"chr{chrom}"]
    refseq_candidate = chromosome_to_refseq.get(chrom)
    if refseq_candidate is not None:
        direct_candidates.append(refseq_candidate)

    for candidate in direct_candidates:
        if candidate in fasta_sequence_name_set:
            return candidate

    if uses_chr_prefix:
        return f"chr{chrom}"
    return chrom


clinvar_complete_df["ChromosomeFASTA"] = clinvar_complete_df["Chromosome"].map(normalize_chromosome_for_fasta)

print(f"FASTA: {FASTA_PATH}")
print("Vi du sequence names trong FASTA:", fasta_sequence_names[:5])
print(f"FASTA co dung prefix chr khong: {uses_chr_prefix}")

display(clinvar_complete_df[["Chromosome", "ChromosomeFASTA"]].drop_duplicates().head(30))
print(f"So dong co ChromosomeFASTA khop FASTA: {clinvar_complete_df['ChromosomeFASTA'].isin(fasta_sequence_name_set).sum():,} / {len(clinvar_complete_df):,}")

FASTA: /mnt/MyData/Bioinformatics/Project/Data/ncbi_dataset/ncbi_dataset/data/GCF_000001405.26/GCF_000001405.26_GRCh38_genomic.fna
Vi du sequence names trong FASTA: ['NC_000001.11', 'NT_187361.1', 'NT_187362.1', 'NT_187363.1', 'NT_187364.1']
FASTA co dung prefix chr khong: False


,Chromosome,ChromosomeFASTA
7,11,11
23,6,6
41,2,2
49,20,NC_000020.11
55,10,NC_000010.11
91,16,16
97,22,NC_000022.11
105,15,15
115,1,NC_000001.11
119,7,7


So dong co ChromosomeFASTA khop FASTA: 460,498 / 1,426,193


## 8. Kiem tra allele goc voi FASTA GRCh38

Voi moi variant:

1. Lay base tai `PositionVCF` tu FASTA GRCh38.
2. So sanh voi `ReferenceAlleleVCF`.
3. Neu khong khop thi loai bo hoac danh dau loi.

Voi SNV, `ReferenceAlleleVCF` nen la mot base `A/C/G/T`. Dataframe sau buoc nay la `clinvar_final_df`.

In [31]:
try:
    from pyfaidx import Fasta
except ImportError as exc:
    raise ImportError(
        "Thieu package pyfaidx trong test-env. Cai bang: conda install -n test-env -c bioconda pyfaidx "
        "hoac pip install pyfaidx"
    ) from exc


fasta = Fasta(str(FASTA_PATH), sequence_always_upper=True)


def get_reference_base(row):
    chrom = row["ChromosomeFASTA"]
    position = int(row["PositionVCF"])
    # ClinVar PositionVCF la 1-based; pyfaidx slicing la 0-based, end-exclusive.
    return str(fasta[chrom][position - 1:position]).upper()


allele_check_df = clinvar_complete_df.copy()
allele_check_df["ReferenceAlleleVCF"] = allele_check_df["ReferenceAlleleVCF"].str.upper()
allele_check_df["AlternateAlleleVCF"] = allele_check_df["AlternateAlleleVCF"].str.upper()

snv_base_mask = (
    allele_check_df["ReferenceAlleleVCF"].str.fullmatch("[ACGT]")
    & allele_check_df["AlternateAlleleVCF"].str.fullmatch("[ACGT]")
)
known_chromosome_mask = allele_check_df["ChromosomeFASTA"].isin(set(fasta.keys()))
position_mask = allele_check_df["PositionVCF"].str.fullmatch(r"\d+")

allele_check_df = allele_check_df.loc[snv_base_mask & known_chromosome_mask & position_mask].copy()
print(f"Kiem tra REF allele voi FASTA cho {len(allele_check_df):,} variants...")
allele_check_df["ReferenceBaseGRCh38"] = allele_check_df.progress_apply(get_reference_base, axis=1)
allele_check_df["ReferenceAlleleMatchesGRCh38"] = allele_check_df["ReferenceBaseGRCh38"].eq(
    allele_check_df["ReferenceAlleleVCF"]
)

clinvar_final_df = allele_check_df.loc[
    allele_check_df["ReferenceAlleleMatchesGRCh38"]
].copy()

print(f"So dong truoc khi kiem tra allele FASTA: {len(clinvar_complete_df):,}")
print(f"So dong SNV co allele/toa do/chromosome hop le de kiem tra: {len(allele_check_df):,}")
print(f"So dong REF khop GRCh38: {len(clinvar_final_df):,}")
print(f"So dong REF khong khop GRCh38: {(~allele_check_df['ReferenceAlleleMatchesGRCh38']).sum():,}")

display(allele_check_df["ReferenceAlleleMatchesGRCh38"].value_counts(dropna=False))
display(
    allele_check_df.loc[~allele_check_df["ReferenceAlleleMatchesGRCh38"], [
        "Chromosome", "ChromosomeFASTA", "PositionVCF", "ReferenceAlleleVCF", "ReferenceBaseGRCh38", "AlternateAlleleVCF", "GeneSymbol"
    ]].head(20)
)

So dong truoc khi kiem tra allele FASTA: 1,426,193
So dong SNV co allele/toa do/chromosome hop le de kiem tra: 460,494
So dong REF khop GRCh38: 460,492
So dong REF khong khop GRCh38: 2


ReferenceAlleleMatchesGRCh38
True     460492
False         2
Name: count, dtype: int64

,Chromosome,ChromosomeFASTA,PositionVCF,ReferenceAlleleVCF,ReferenceBaseGRCh38,AlternateAlleleVCF,GeneSymbol
4710295,17,NC_000017.11,268116,G,A,C,CCL4L2
6446665,22,NC_000022.11,273613,G,N,A,GSTT1


## 8B. Luu Parquet trung gian sau khi loc sach

Luu `clinvar_final_df` sau cac buoc loc 1-8 vao file Parquet de lam dau vao cho buoc 9 lay DNA context.

File output:

`processed/clinvar_filtered_step8.parquet`

In [32]:
processed_dir = PROJECT_DIR / "processed"
processed_dir.mkdir(exist_ok=True)

step8_parquet_path = processed_dir / "clinvar_filtered_step8.parquet"

clinvar_final_df.to_parquet(step8_parquet_path, index=False, engine="pyarrow")

print(f"Saved: {step8_parquet_path}")
print(f"Rows: {len(clinvar_final_df):,}")
print(f"Columns: {len(clinvar_final_df.columns):,}")

# Doc lai nhanh de kiem tra file Parquet hop le.
step8_check_df = pd.read_parquet(step8_parquet_path, engine="pyarrow")
print(f"Read back shape: {step8_check_df.shape}")
step8_check_df.head()

Saved: /mnt/MyData/Bioinformatics/Project/processed/clinvar_filtered_step8.parquet
Rows: 460,492
Columns: 13
Read back shape: (460492, 13)


,Type,GeneSymbol,ClinicalSignificance,Assembly,Chromosome,ReviewStatus,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,Y,ChromosomeFASTA,ReferenceBaseGRCh38,ReferenceAlleleMatchesGRCh38
0,single nucleotide variant,ABHD12,Pathogenic,GRCh38,20,"criteria provided, multiple submitters, no conflicts",25302322,G,A,1,NC_000020.11,G,True
1,single nucleotide variant,HOGA1,Pathogenic,GRCh38,10,"criteria provided, multiple submitters, no conflicts",97611535,G,T,1,NC_000010.11,G,True
2,single nucleotide variant,HOGA1,Pathogenic/Likely pathogenic,GRCh38,10,"criteria provided, multiple submitters, no conflicts",97598852,C,T,1,NC_000010.11,C,True
3,single nucleotide variant,HOGA1,Pathogenic/Likely pathogenic,GRCh38,10,"criteria provided, multiple submitters, no conflicts",97584912,G,C,1,NC_000010.11,G,True
4,single nucleotide variant,HOGA1,Pathogenic/Likely pathogenic,GRCh38,10,"criteria provided, multiple submitters, no conflicts",97601925,T,G,1,NC_000010.11,T,True


## 9. Lay chuoi DNA context quanh mutation

Voi moi variant trong `clinvar_final_df`, lay:

- 100 bp truoc mutation
- 1 bp tai vi tri mutation
- 100 bp sau mutation

Tong do dai sequence context la `201 bp`.

Cot moi:

- `SequenceContext201`: chuoi DNA 201 bp
- `ContextStart1Based`: toa do bat dau context theo he 1-based
- `ContextEnd1Based`: toa do ket thuc context theo he 1-based

Sau buoc nay, dataframe `clinvar_context_df` se la dau vao tot hon cho buoc one-hot encode/CNN.

In [34]:
from pyfaidx import Fasta

CONTEXT_FLANK_BP = 100
CONTEXT_LENGTH = CONTEXT_FLANK_BP * 2 + 1

# Tao lai FASTA object neu kernel chua co bien `fasta`.
if "fasta" not in globals():
    fasta = Fasta(str(FASTA_PATH), sequence_always_upper=True)


def extract_sequence_context(row, flank=CONTEXT_FLANK_BP):
    chrom = row["ChromosomeFASTA"]
    position_1based = int(row["PositionVCF"])

    start_0based = position_1based - 1 - flank
    end_0based_exclusive = position_1based + flank

    if start_0based < 0:
        return pd.NA
    if chrom not in fasta:
        return pd.NA
    if end_0based_exclusive > len(fasta[chrom]):
        return pd.NA

    sequence = str(fasta[chrom][start_0based:end_0based_exclusive]).upper()
    return sequence


clinvar_context_df = clinvar_final_df.copy()
clinvar_context_df["ContextStart1Based"] = clinvar_context_df["PositionVCF"].astype(int) - CONTEXT_FLANK_BP
clinvar_context_df["ContextEnd1Based"] = clinvar_context_df["PositionVCF"].astype(int) + CONTEXT_FLANK_BP
print(f"Lay sequence context 201 bp cho {len(clinvar_context_df):,} variants...")
clinvar_context_df["SequenceContext201"] = clinvar_context_df.progress_apply(extract_sequence_context, axis=1)

valid_context_mask = clinvar_context_df["SequenceContext201"].notna()
valid_context_mask &= clinvar_context_df["SequenceContext201"].str.len().eq(CONTEXT_LENGTH)
valid_context_mask &= clinvar_context_df["SequenceContext201"].str.fullmatch("[ACGTN]+")

clinvar_context_df = clinvar_context_df.loc[valid_context_mask].copy()

clinvar_context_df["ContextCenterBase"] = clinvar_context_df["SequenceContext201"].str[CONTEXT_FLANK_BP]
center_ref_mask = clinvar_context_df["ContextCenterBase"].eq(clinvar_context_df["ReferenceAlleleVCF"])

print(f"So dong truoc khi lay context: {len(clinvar_final_df):,}")
print(f"So dong co context dai dung {CONTEXT_LENGTH} bp: {len(clinvar_context_df):,}")
print(f"So dong co base giua context khop REF: {center_ref_mask.sum():,}")
print(f"So dong base giua context khong khop REF: {(~center_ref_mask).sum():,}")

clinvar_context_df = clinvar_context_df.loc[center_ref_mask].copy()

print(f"So dong sau khi giu context hop le: {len(clinvar_context_df):,}")

display(clinvar_context_df["SequenceContext201"].str.len().value_counts(dropna=False))
display(clinvar_context_df[["ReferenceAlleleVCF", "ContextCenterBase"]].value_counts().head(20))

So dong truoc khi lay context: 460,492
So dong co context dai dung 201 bp: 460,491
So dong co base giua context khop REF: 460,491
So dong base giua context khong khop REF: 0
So dong sau khi giu context hop le: 460,491


SequenceContext201
201    460491
Name: count, dtype: int64

ReferenceAlleleVCF  ContextCenterBase
C                   C                    155701
G                   G                    155214
T                   T                     75255
A                   A                     74321
Name: count, dtype: int64

## 9B. Luu Parquet co DNA context 201 bp

Luu `clinvar_context_df` de dung cho buoc one-hot encode thanh tensor CNN.

In [35]:
context_parquet_path = processed_dir / "clinvar_context_201.parquet"

clinvar_context_df.to_parquet(context_parquet_path, index=False, engine="pyarrow")

print(f"Saved: {context_parquet_path}")
print(f"Rows: {len(clinvar_context_df):,}")
print(f"Columns: {len(clinvar_context_df.columns):,}")

context_check_df = pd.read_parquet(context_parquet_path, engine="pyarrow")
print(f"Read back shape: {context_check_df.shape}")
context_check_df.head()

Saved: /mnt/MyData/Bioinformatics/Project/processed/clinvar_context_201.parquet
Rows: 460,491
Columns: 17
Read back shape: (460491, 17)


,Type,GeneSymbol,ClinicalSignificance,Assembly,Chromosome,ReviewStatus,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,Y,ChromosomeFASTA,ReferenceBaseGRCh38,ReferenceAlleleMatchesGRCh38,ContextStart1Based,ContextEnd1Based,SequenceContext201,ContextCenterBase
0,single nucleotide variant,ABHD12,Pathogenic,GRCh38,20,"criteria provided, multiple submitters, no conflicts",25302322,G,A,1,NC_000020.11,G,True,25302222,25302422,AGTATCCGTGGCAGCTCAGGGCTCTTGTAAATGTATTTGTGCCTGTAGCCAAGGTCTGAATGAAAGGGCACAAACTGAACTTTGAAATCTCGGAAGCTTCGAGCTGGTGCGGCGAT...,G
1,single nucleotide variant,HOGA1,Pathogenic,GRCh38,10,"criteria provided, multiple submitters, no conflicts",97611535,G,T,1,NC_000010.11,G,True,97611435,97611635,GTGACAGATCTCCGAGTTCCAGATATGGGTGTTCAGCAAATTTCTCAGTCTCTTCTAACAGGCCCTGCTTTGCAGGTGACCCGGCGCTTTGGGATCCCAGGGCTGAAGAAAATCAT...,G
2,single nucleotide variant,HOGA1,Pathogenic/Likely pathogenic,GRCh38,10,"criteria provided, multiple submitters, no conflicts",97598852,C,T,1,NC_000010.11,C,True,97598752,97598952,TTAGTCAGCTGTGTCTCTTGCAGGCTTCGTGGTCCAGGGCTCCAATGGCGAGTTTCCTTTCCTGACCAGCAGTGAGCGCCTCGAGGTGGTGAGCCGTGTGCGCCAGGCCATGCCCA...,C
3,single nucleotide variant,HOGA1,Pathogenic/Likely pathogenic,GRCh38,10,"criteria provided, multiple submitters, no conflicts",97584912,G,C,1,NC_000010.11,G,True,97584812,97585012,GGTATCTACCCCCCTGTGACCACCCCCTTCACTGCCACTGCAGAGGTGGACTATGGGAAACTGGAGGAGAATCTGCACAAACTGGGCACCTTCCCCTTCCGAGGTAAGTGGGGCTG...,G
4,single nucleotide variant,HOGA1,Pathogenic/Likely pathogenic,GRCh38,10,"criteria provided, multiple submitters, no conflicts",97601925,T,G,1,NC_000010.11,T,True,97601825,97602025,CTGGCTGATGTTCTGCGTCTTACTTCGTGCAGGAGCTGTGGGGGGCGTCTGCGCCCTGGCCAATGTCCTGGGGGCTCAGGTGTGCCAGCTGGAGCGACTGTGCTGCACGGGGCAAT...,T


## 10. Tao sequence goc va sequence dot bien

Tu `SequenceContext201`, tao hai phien ban:

- `ref_seq`: chuoi goc tu GRCh38
- `alt_seq`: chuoi da thay base o giua bang `AlternateAlleleVCF`

Voi demo CNN don gian, ta se bat dau train bang `alt_seq`. Van giu `ref_seq` trong metadata de sau nay co the thu mo hinh hoc ca cap ref/alt.

In [36]:
MODEL_SEQUENCE_COLUMN = "alt_seq"
CENTER_INDEX = CONTEXT_FLANK_BP

sequence_pair_df = clinvar_context_df.copy()
sequence_pair_df["ref_seq"] = sequence_pair_df["SequenceContext201"].str.upper()


def make_alt_sequence(row):
    ref_seq = row["ref_seq"]
    alt_base = row["AlternateAlleleVCF"]
    return ref_seq[:CENTER_INDEX] + alt_base + ref_seq[CENTER_INDEX + 1:]


print(f"Tao alt_seq cho {len(sequence_pair_df):,} variants...")
sequence_pair_df["alt_seq"] = sequence_pair_df.progress_apply(make_alt_sequence, axis=1)

sequence_pair_df["ref_center_base"] = sequence_pair_df["ref_seq"].str[CENTER_INDEX]
sequence_pair_df["alt_center_base"] = sequence_pair_df["alt_seq"].str[CENTER_INDEX]

print(f"So dong dau vao: {len(clinvar_context_df):,}")
print(f"ref center khop REF: {sequence_pair_df['ref_center_base'].eq(sequence_pair_df['ReferenceAlleleVCF']).sum():,}")
print(f"alt center khop ALT: {sequence_pair_df['alt_center_base'].eq(sequence_pair_df['AlternateAlleleVCF']).sum():,}")

display(sequence_pair_df[[
    "ReferenceAlleleVCF", "AlternateAlleleVCF", "ref_center_base", "alt_center_base", "ref_seq", "alt_seq"
]].head(10))

So dong dau vao: 460,491
ref center khop REF: 460,491
alt center khop ALT: 460,491


,ReferenceAlleleVCF,AlternateAlleleVCF,ref_center_base,alt_center_base,ref_seq,alt_seq
49,G,A,G,A,AGTATCCGTGGCAGCTCAGGGCTCTTGTAAATGTATTTGTGCCTGTAGCCAAGGTCTGAATGAAAGGGCACAAACTGAACTTTGAAATCTCGGAAGCTTCGAGCTGGTGCGGCGAT...,AGTATCCGTGGCAGCTCAGGGCTCTTGTAAATGTATTTGTGCCTGTAGCCAAGGTCTGAATGAAAGGGCACAAACTGAACTTTGAAATCTCGGAAGCTTCAAGCTGGTGCGGCGAT...
55,G,T,G,T,GTGACAGATCTCCGAGTTCCAGATATGGGTGTTCAGCAAATTTCTCAGTCTCTTCTAACAGGCCCTGCTTTGCAGGTGACCCGGCGCTTTGGGATCCCAGGGCTGAAGAAAATCAT...,GTGACAGATCTCCGAGTTCCAGATATGGGTGTTCAGCAAATTTCTCAGTCTCTTCTAACAGGCCCTGCTTTGCAGGTGACCCGGCGCTTTGGGATCCCAGTGCTGAAGAAAATCAT...
57,C,T,C,T,TTAGTCAGCTGTGTCTCTTGCAGGCTTCGTGGTCCAGGGCTCCAATGGCGAGTTTCCTTTCCTGACCAGCAGTGAGCGCCTCGAGGTGGTGAGCCGTGTGCGCCAGGCCATGCCCA...,TTAGTCAGCTGTGTCTCTTGCAGGCTTCGTGGTCCAGGGCTCCAATGGCGAGTTTCCTTTCCTGACCAGCAGTGAGCGCCTCGAGGTGGTGAGCCGTGTGTGCCAGGCCATGCCCA...
61,G,C,G,C,GGTATCTACCCCCCTGTGACCACCCCCTTCACTGCCACTGCAGAGGTGGACTATGGGAAACTGGAGGAGAATCTGCACAAACTGGGCACCTTCCCCTTCCGAGGTAAGTGGGGCTG...,GGTATCTACCCCCCTGTGACCACCCCCTTCACTGCCACTGCAGAGGTGGACTATGGGAAACTGGAGGAGAATCTGCACAAACTGGGCACCTTCCCCTTCCCAGGTAAGTGGGGCTG...
63,T,G,T,G,CTGGCTGATGTTCTGCGTCTTACTTCGTGCAGGAGCTGTGGGGGGCGTCTGCGCCCTGGCCAATGTCCTGGGGGCTCAGGTGTGCCAGCTGGAGCGACTGTGCTGCACGGGGCAAT...,CTGGCTGATGTTCTGCGTCTTACTTCGTGCAGGAGCTGTGGGGGGCGTCTGCGCCCTGGCCAATGTCCTGGGGGCTCAGGTGTGCCAGCTGGAGCGACTGGGCTGCACGGGGCAAT...
97,G,T,G,T,TCATCATGTTGGCCACTACCTCGGGATGGATGTCCATGACACTCCAGACATGCCCCGTTCCCTCCCTCTGCAGCCTGGGATGGTAATCACAATTGAGCCCGGTAAGGAGAGGTGTT...,TCATCATGTTGGCCACTACCTCGGGATGGATGTCCATGACACTCCAGACATGCCCCGTTCCCTCCCTCTGCAGCCTGGGATGGTAATCACAATTGAGCCCTGTAAGGAGAGGTGTT...
115,C,T,C,T,GCGGTGGCTCACGCCTGTAATCCCAGCACTTTGGGAGGCTGAAGCGGGCGAATCACGAGGTCAAGAGATTGAGACCATCCTGGCCAACATGGTGAAACCCCGTTTCTACTAAAAAT...,GCGGTGGCTCACGCCTGTAATCCCAGCACTTTGGGAGGCTGAAGCGGGCGAATCACGAGGTCAAGAGATTGAGACCATCCTGGCCAACATGGTGAAACCCTGTTTCTACTAAAAAT...
117,A,T,A,T,TAAAAAGAAAAAAATTGAGTTTAATATTAAAAATTAAAGTTTACTTATAAAATACAGGACATAGAGAAAAATGTACTTCTATTTTTCTTTTCTATAGGAGAAGCTAAAACTTACTT...,TAAAAAGAAAAAAATTGAGTTTAATATTAAAAATTAAAGTTTACTTATAAAATACAGGACATAGAGAAAAATGTACTTCTATTTTTCTTTTCTATAGGAGTAGCTAAAACTTACTT...
125,G,T,G,T,GCCCTCTGTAGCCTGAGATCTGCTTTTTTCTAGATCATCTTTGCTAAGGATGGGCATTTTGCCCTGGAGGAGCTGGCCCAAGCTGGCTATGAGGTGGTTGGGCTTGACTGGACAGT...,GCCCTCTGTAGCCTGAGATCTGCTTTTTTCTAGATCATCTTTGCTAAGGATGGGCATTTTGCCCTGGAGGAGCTGGCCCAAGCTGGCTATGAGGTGGTTGTGCTTGACTGGACAGT...
127,G,A,G,A,GCCCTCTGTAGCCTGAGATCTGCTTTTTTCTAGATCATCTTTGCTAAGGATGGGCATTTTGCCCTGGAGGAGCTGGCCCAAGCTGGCTATGAGGTGGTTGGGCTTGACTGGACAGT...,GCCCTCTGTAGCCTGAGATCTGCTTTTTTCTAGATCATCTTTGCTAAGGATGGGCATTTTGCCCTGGAGGAGCTGGCCCAAGCTGGCTATGAGGTGGTTGAGCTTGACTGGACAGT...


## 11. Loai sequence chua ky tu la

Chi giu sequence gom bon base chuan:

- `A`
- `C`
- `G`
- `T`

Loai cac sequence co `N`, `R`, `Y`, `K`, `M`, ... vi one-hot encoding ban dau chi xu ly 4 base.

In [37]:
valid_base_pattern = "^[ACGT]+$"

valid_ref_mask = sequence_pair_df["ref_seq"].str.fullmatch(valid_base_pattern)
valid_alt_mask = sequence_pair_df["alt_seq"].str.fullmatch(valid_base_pattern)
valid_length_mask = sequence_pair_df["alt_seq"].str.len().eq(CONTEXT_LENGTH)

sequence_model_df = sequence_pair_df.loc[
    valid_ref_mask & valid_alt_mask & valid_length_mask
].copy()

print(f"So dong truoc khi loc ky tu la: {len(sequence_pair_df):,}")
print(f"So dong sau khi chi giu A/C/G/T va length {CONTEXT_LENGTH}: {len(sequence_model_df):,}")
print(f"So dong bi loai: {len(sequence_pair_df) - len(sequence_model_df):,}")

display(sequence_model_df[MODEL_SEQUENCE_COLUMN].str.len().value_counts(dropna=False))
display(sequence_model_df[MODEL_SEQUENCE_COLUMN].str.contains("[^ACGT]", regex=True).value_counts(dropna=False))

So dong truoc khi loc ky tu la: 460,491
So dong sau khi chi giu A/C/G/T va length 201: 460,488
So dong bi loai: 3


alt_seq
201    460488
Name: count, dtype: int64

alt_seq
False    460488
Name: count, dtype: int64

## 12. One-hot encoding sequence

Ma hoa DNA sequence thanh ma tran so:

- `A = [1, 0, 0, 0]`
- `C = [0, 1, 0, 0]`
- `G = [0, 0, 1, 0]`
- `T = [0, 0, 0, 1]`

Sequence dai `201 bp` se thanh tensor kich thuoc `201 x 4`.

Voi demo CNN dau tien, tensor `X_alt` duoc tao tu `alt_seq`.

In [38]:
import numpy as np

base_to_index = {"A": 0, "C": 1, "G": 2, "T": 3}


def one_hot_encode_sequence(sequence):
    encoded = np.zeros((len(sequence), 4), dtype=np.uint8)
    for position, base in enumerate(sequence):
        encoded[position, base_to_index[base]] = 1
    return encoded


print(f"One-hot encode {len(sequence_model_df):,} sequences...")
X_alt = np.stack([
    one_hot_encode_sequence(sequence)
    for sequence in tqdm(sequence_model_df[MODEL_SEQUENCE_COLUMN], desc="one-hot", unit="seq")
])
y = sequence_model_df["Y"].to_numpy(dtype=np.int8)

print(f"X_alt shape: {X_alt.shape}")
print(f"y shape: {y.shape}")
print(f"X_alt dtype: {X_alt.dtype}")
print(f"y dtype: {y.dtype}")
print(f"Label counts: {dict(zip(*np.unique(y, return_counts=True)))}")

# Kiem tra moi position chi co dung mot base duoc bat.
print(f"One-hot hop le: {np.all(X_alt.sum(axis=2) == 1)}")

X_alt shape: (460488, 201, 4)
y shape: (460488,)
X_alt dtype: uint8
y dtype: int8
Label counts: {np.int8(0): np.int64(401054), np.int8(1): np.int64(59434)}
One-hot hop le: True


## 12A. Tao tensor ref/alt/marker 9-channel

Dung y tuong `4 + 4 + 1`:

- 4 channels cho `ref_seq`: A/C/G/T
- 4 channels cho `alt_seq`: A/C/G/T
- 1 channel danh dau vi tri mutation o giua sequence

Tensor moi co shape:

`(n_variants, 201, 9)`

Channel marker bang `1` tai index `100`, cac vi tri khac bang `0`.

In [ ]:
def one_hot_encode_sequence_uint8(sequence):
    encoded = np.zeros((len(sequence), 4), dtype=np.uint8)
    for position, base in enumerate(sequence):
        encoded[position, base_to_index[base]] = 1
    return encoded


print(f"Tao tensor ref/alt/marker cho {len(sequence_model_df):,} sequences...")
X_ref_alt_marker = np.zeros((len(sequence_model_df), CONTEXT_LENGTH, 9), dtype=np.uint8)

for row_idx, row in enumerate(tqdm(sequence_model_df.itertuples(index=False), total=len(sequence_model_df), desc="ref-alt-marker", unit="seq")):
    ref_encoded = one_hot_encode_sequence_uint8(row.ref_seq)
    alt_encoded = one_hot_encode_sequence_uint8(row.alt_seq)
    X_ref_alt_marker[row_idx, :, 0:4] = ref_encoded
    X_ref_alt_marker[row_idx, :, 4:8] = alt_encoded
    X_ref_alt_marker[row_idx, CENTER_INDEX, 8] = 1

print(f"X_ref_alt_marker shape: {X_ref_alt_marker.shape}")
print(f"X_ref_alt_marker dtype: {X_ref_alt_marker.dtype}")
print(f"Marker channel hop le: {np.all(X_ref_alt_marker[:, CENTER_INDEX, 8] == 1) and np.all(X_ref_alt_marker[:, :CENTER_INDEX, 8] == 0) and np.all(X_ref_alt_marker[:, CENTER_INDEX + 1:, 8] == 0)}")
print(f"Ref one-hot hop le: {np.all(X_ref_alt_marker[:, :, 0:4].sum(axis=2) == 1)}")
print(f"Alt one-hot hop le: {np.all(X_ref_alt_marker[:, :, 4:8].sum(axis=2) == 1)}")

## 12B. Luu du lieu san sang train

Sau buoc 10-12, du lieu da san sang cho demo CNN.

Output:

- `processed/X_alt_201.npy`: tensor one-hot cua `alt_seq`, shape `(n_variants, 201, 4)`
- `processed/X_ref_alt_marker_201.npy`: tensor 9-channel `ref + alt + marker`, shape `(n_variants, 201, 9)`
- `processed/y.npy`: nhan `0/1`
- `processed/clinvar_training_metadata.parquet`: metadata tuong ung tung dong trong tensor va `y`
- `processed/clinvar_cnn_dataset_ref_alt_marker_201.npz`: file gom `X_ref_alt_marker` va `y` de load nhanh

In [39]:
x_alt_path = processed_dir / "X_alt_201.npy"
x_ref_alt_marker_path = processed_dir / "X_ref_alt_marker_201.npy"
y_path = processed_dir / "y.npy"
metadata_path = processed_dir / "clinvar_training_metadata.parquet"
npz_alt_path = processed_dir / "clinvar_cnn_dataset_alt_201.npz"
npz_ref_alt_marker_path = processed_dir / "clinvar_cnn_dataset_ref_alt_marker_201.npz"

np.save(x_alt_path, X_alt)
np.save(x_ref_alt_marker_path, X_ref_alt_marker)
np.save(y_path, y)

metadata_columns = [
    "GeneSymbol",
    "ClinicalSignificance",
    "Y",
    "Assembly",
    "Type",
    "ReviewStatus",
    "Chromosome",
    "ChromosomeFASTA",
    "PositionVCF",
    "ReferenceAlleleVCF",
    "AlternateAlleleVCF",
    "ContextStart1Based",
    "ContextEnd1Based",
    "ref_seq",
    "alt_seq",
]
sequence_model_df[metadata_columns].to_parquet(metadata_path, index=False, engine="pyarrow")
np.savez_compressed(npz_alt_path, X_alt=X_alt, y=y)
np.savez_compressed(npz_ref_alt_marker_path, X_ref_alt_marker=X_ref_alt_marker, y=y)

print(f"Saved X_alt: {x_alt_path}")
print(f"Saved X_ref_alt_marker: {x_ref_alt_marker_path}")
print(f"Saved y: {y_path}")
print(f"Saved metadata: {metadata_path}")
print(f"Saved alt dataset: {npz_alt_path}")
print(f"Saved ref/alt/marker dataset: {npz_ref_alt_marker_path}")

X_alt_check = np.load(x_alt_path)
X_ref_alt_marker_check = np.load(x_ref_alt_marker_path)
y_check = np.load(y_path)
metadata_check = pd.read_parquet(metadata_path, engine="pyarrow")

print(f"X_alt_check shape: {X_alt_check.shape}")
print(f"X_ref_alt_marker_check shape: {X_ref_alt_marker_check.shape}")
print(f"y_check shape: {y_check.shape}")
print(f"metadata_check shape: {metadata_check.shape}")

Saved X_alt: /mnt/MyData/Bioinformatics/Project/processed/X_alt_201.npy
Saved y: /mnt/MyData/Bioinformatics/Project/processed/y.npy
Saved metadata: /mnt/MyData/Bioinformatics/Project/processed/clinvar_training_metadata.parquet
Saved compressed dataset: /mnt/MyData/Bioinformatics/Project/processed/clinvar_cnn_dataset_alt_201.npz
X_check shape: (460488, 201, 4)
y_check shape: (460488,)
metadata_check shape: (460488, 15)


## Demo vai chuc dong dau cua dataset train

`sequence_model_df`, `X_alt`, va `y` la dau ra sau buoc 12.

In [33]:
demo_columns = [
    "GeneSymbol",
    "ClinicalSignificance",
    "Y",
    "Assembly",
    "Type",
    "ReviewStatus",
    "Chromosome",
    "ChromosomeFASTA",
    "PositionVCF",
    "ReferenceAlleleVCF",
    "ReferenceBaseGRCh38",
    "AlternateAlleleVCF",
    "ContextStart1Based",
    "ContextEnd1Based",
    "SequenceContext201",
    "ref_seq",
    "alt_seq",
]

sequence_model_df[demo_columns].head(30)

,GeneSymbol,ClinicalSignificance,Y,Assembly,Type,ReviewStatus,Chromosome,ChromosomeFASTA,PositionVCF,ReferenceAlleleVCF,ReferenceBaseGRCh38,AlternateAlleleVCF
49,ABHD12,Pathogenic,1,GRCh38,single nucleotide variant,"criteria provided, multiple submitters, no conflicts",20,NC_000020.11,25302322,G,G,A
55,HOGA1,Pathogenic,1,GRCh38,single nucleotide variant,"criteria provided, multiple submitters, no conflicts",10,NC_000010.11,97611535,G,G,T
57,HOGA1,Pathogenic/Likely pathogenic,1,GRCh38,single nucleotide variant,"criteria provided, multiple submitters, no conflicts",10,NC_000010.11,97598852,C,C,T
61,HOGA1,Pathogenic/Likely pathogenic,1,GRCh38,single nucleotide variant,"criteria provided, multiple submitters, no conflicts",10,NC_000010.11,97584912,G,G,C
63,HOGA1,Pathogenic/Likely pathogenic,1,GRCh38,single nucleotide variant,"criteria provided, multiple submitters, no conflicts",10,NC_000010.11,97601925,T,T,G
97,XPNPEP3,Pathogenic,1,GRCh38,single nucleotide variant,"criteria provided, single submitter",22,NC_000022.11,40924482,G,G,T
115,SDCCAG8,Pathogenic/Likely pathogenic,1,GRCh38,single nucleotide variant,"criteria provided, multiple submitters, no conflicts",1,NC_000001.11,243305133,C,C,T
117,SDCCAG8,Pathogenic,1,GRCh38,single nucleotide variant,"criteria provided, single submitter",1,NC_000001.11,243304716,A,A,T
125,UROD,Pathogenic/Likely pathogenic,1,GRCh38,single nucleotide variant,"criteria provided, multiple submitters, no conflicts",1,NC_000001.11,45014803,G,G,T
127,UROD,Pathogenic,1,GRCh38,single nucleotide variant,"criteria provided, single submitter",1,NC_000001.11,45014803,G,G,A
